# 01 — Data Loading & Exploratory Data Analysis (BAFU 2219)

Loads the BAFU/FOEN daily maximum discharge record for the Simme at Oberried/Lenk
(station 2219, `Tagesmaxima`), performs basic quality checks, extracts the Annual
Maximum Series (AMS) for 1974–2025, and exports `data/processed/annual_summary_2219.csv`
for use by downstream notebooks.

**Key differences from the GRDC analysis (notebooks/):**
- Source: BAFU/FOEN station 2219 (daily maxima) instead of GRDC daily means
- Period: 1974–2025 (n=52) instead of 1944–2020 (n=77); pre-GLOF reference is 1974–2010 (n=37)
- 2026 is dropped (partial year, data only through May)

| # | Step | Lecture Reference |
|---|------|------------------|
| 1 | Raw discharge loading & QC | — |
| 2 | GLOF event catalogue | — |
| 3 | Annual block maxima extraction | Module 2, §4.1 |
| 4 | Descriptive statistics | Module 1, §1.2 |
| 5 | Time series + annual maxima plot | — |
| 6 | Export processed data | — |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from utils import load_2219_discharge, extract_annual_maxima, GLOF_DATES, GLOF_WINDOW

FIGS     = Path('../figures_2219/01_eda')
DATA_OUT = Path('../data/processed')
FIGS.mkdir(parents=True, exist_ok=True)
DATA_OUT.mkdir(exist_ok=True)

## 1 — Raw Discharge Loading & Quality Control

In [ ]:
Q_daily_raw = load_2219_discharge()

# Drop partial year 2026
Q_daily = Q_daily_raw[Q_daily_raw.index.year <= 2025]

print(f'Station:  BAFU/FOEN 2219 — Simme at Oberried/Lenk')
print(f'Series:   Tagesmaxima (daily maximum discharge)')
print(f'Period:   {Q_daily.index[0].date()}  to  {Q_daily.index[-1].date()}')
print(f'Days:     {len(Q_daily):,}  |  NaN: {Q_daily.isna().sum()}')
print(f'Range:    {Q_daily.min():.2f} – {Q_daily.max():.1f} m³/s  |  mean: {Q_daily.mean():.2f} m³/s')
if Q_daily_raw.index.year.max() == 2026:
    n_dropped = (Q_daily_raw.index.year == 2026).sum()
    print(f'\nNote: dropped {n_dropped} records from partial year 2026.')

## 2 — GLOF Event Catalogue

Confirmed outburst dates from gletschersee-lenk.ch, Simmentalzeitung, 20min.ch, and TC (2021).
2016 excluded: no documented major outburst. Each event is masked with a ±5-day window
in the homogeneity analysis (Notebook 03).

In [ ]:
print(f'GLOF events documented ({len(GLOF_DATES)} years, ±{GLOF_WINDOW}-day window):')
for yr, dt in GLOF_DATES.items():
    q_on_date = Q_daily.get(dt, float('nan'))
    print(f'  {yr}: {dt.date()}   Q on GLOF day = {q_on_date:.2f} m³/s')

## 3 — Annual Block Maxima (Module 2, §4.1)

Since the source data is already `Tagesmaxima` (daily maxima), block maxima extraction
simply takes the maximum of each calendar year's daily maxima — exactly equivalent to
applying block maxima to the underlying sub-daily record.

In [ ]:
Q_ann = extract_annual_maxima(Q_daily)
Q_ann_date = Q_daily.resample('YE').apply(lambda s: s.idxmax())
Q_ann_date.index = Q_ann_date.index.year

glof_years = set(GLOF_DATES.keys())

annual = pd.DataFrame({
    'Q_max':      Q_ann,
    'Q_max_date': Q_ann_date,
    'is_glof':    [yr in glof_years for yr in Q_ann.index],
})
annual.index.name = 'year'

n = len(annual)
print(f'Annual Maximum Series: n={n} years  ({annual.index[0]}–{annual.index[-1]})')
print(f'GLOF years in AMS: {sorted(glof_years & set(annual.index))}')

## 4 — Descriptive Statistics (Module 1, §1.2)

In [ ]:
Q = annual['Q_max']
Q_no_glof = Q[~annual['is_glof']]
Q_glof    = Q[ annual['is_glof']]

print(f'                   Full series   Non-GLOF years   GLOF years')
print(f'n                  {len(Q):>11}   {len(Q_no_glof):>14}   {len(Q_glof):>10}')
print(f'Mean (m³/s)        {Q.mean():>11.2f}   {Q_no_glof.mean():>14.2f}   {Q_glof.mean():>10.2f}')
print(f'Std  (m³/s)        {Q.std():>11.2f}   {Q_no_glof.std():>14.2f}   {Q_glof.std():>10.2f}')
print(f'Median (m³/s)      {Q.median():>11.2f}   {Q_no_glof.median():>14.2f}   {Q_glof.median():>10.2f}')
print(f'Min / Max (m³/s)   {Q.min():.2f} / {Q.max():.2f}   {Q_no_glof.min():.2f} / {Q_no_glof.max():.2f}   {Q_glof.min():.2f} / {Q_glof.max():.2f}')

print('\nTop 10 annual maxima:')
print(annual['Q_max'].nlargest(10).to_frame().join(annual[['Q_max_date', 'is_glof']]).to_string())

## 5 — Time Series & Annual Maxima Plot

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# Panel A: daily discharge
ax = axes[0]
ax.plot(Q_daily.index, Q_daily.values, lw=0.4, color='steelblue')
for dt in GLOF_DATES.values():
    ax.axvline(dt, color='crimson', lw=0.8, alpha=0.7)
ax.set_ylabel('Q (m³/s)')
ax.set_title('Simme at Lenk (BAFU 2219, Tagesmaxima) — daily max discharge 1974–2025\n(red lines = confirmed GLOF dates)')
ax.grid(alpha=0.25)

# Panel B: annual maxima bar chart
ax2 = axes[1]
glof_mask = annual['is_glof']
ax2.bar(annual.index, annual['Q_max'], color='steelblue', alpha=0.65, label='Annual max Q')
ax2.bar(
    annual.index[glof_mask], annual.loc[glof_mask, 'Q_max'],
    color='crimson', alpha=0.9, label='GLOF year'
)
for yr in annual.index[glof_mask]:
    ax2.text(yr, annual.loc[yr, 'Q_max'] + 0.25, str(yr),
             ha='center', va='bottom', fontsize=7.5, color='crimson')
ax2.set_xlabel('Year')
ax2.set_ylabel('Q_max (m³/s)')
ax2.set_title('Annual Maximum Series — GLOF years highlighted')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(FIGS / '01_discharge_overview.png', dpi=150)
plt.show()
print('Saved -> figures_2219/01_eda/01_discharge_overview.png')

## 6 — Export Processed Data

In [ ]:
annual.to_csv(DATA_OUT / 'annual_summary_2219.csv')
print(f'Saved -> data/processed/annual_summary_2219.csv  ({len(annual)} rows)')